<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab4_sentiment_analysis_rnn_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — Sentiment Analysis with RNNs/LSTMs (PyTorch)

**Module 4: Introduction to Natural Language Processing**

In this lab you will train a recurrent neural network to decide whether a movie review
is **positive or negative**, using the classic **IMDB** dataset (25k labeled reviews).

```
review (word ids) --> [Embedding] --> [LSTM] --> last hidden state --> [Linear] --> P(positive)
```

You will also build a **bag-of-embeddings baseline** (no recurrence — just average the
word vectors). Comparing the two tells you how much *word order* actually matters, and
where recurrence starts to struggle (long reviews — a preview of why Module 5 introduces
attention).

## What you will do
You only need to fill in **five** short pieces of code, each marked with a `TODO`:
  1. **pad/truncate** integer sequences to a fixed length
  2. the **LSTM classifier** (`__init__` + `forward`)
  3. the **bag-of-embeddings baseline** model
  4. one **training step**
  5. the **evaluation** function (accuracy)

Everything else — data download, vocab decoding, the training driver, plots, and the
long-vs-short review analysis — is provided.

## How to use this file
* Recommended: **Google Colab** with a GPU runtime (CPU works; ~5-10 min).
* Fill in each `TODO`, then run cells top to bottom.

## Setup — imports and configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
VOCAB_SIZE = 10_000   # keep the 10k most frequent words
MAX_LEN = 200         # pad/truncate every review to 200 tokens
EMBED_DIM = 64
HIDDEN_DIM = 64
BATCH_SIZE = 64
EPOCHS = 3
TRAIN_SUBSET = 10_000
TEST_SUBSET = 5_000

np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Section 1 — The data *(provided)*

We load IMDB through `keras.datasets`: each review is already an **integer sequence**
(word id 1 = most frequent word). Ids 0/1/2 are reserved for padding / start / unknown.

In [ ]:
from tensorflow import keras  # used ONLY to download the pre-tokenized dataset

(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=VOCAB_SIZE)

rng = np.random.RandomState(SEED)
itr = rng.choice(len(x_train), TRAIN_SUBSET, replace=False)
ite = rng.choice(len(x_test), TEST_SUBSET, replace=False)
x_train, y_train = x_train[itr], y_train[itr]
x_test, y_test = x_test[ite], y_test[ite]

word_index = keras.datasets.imdb.get_word_index()
id_to_word = {i + 3: w for w, i in word_index.items()}
id_to_word.update({0: "<pad>", 1: "<start>", 2: "<unk>"})

def decode(seq):
    return " ".join(id_to_word.get(i, "?") for i in seq)

print("a raw review (ids):", x_train[0][:12], "...")
print("decoded          :", decode(x_train[0][:12]), "...")
print("label            :", "positive" if y_train[0] == 1 else "negative")
lengths = [len(s) for s in x_train]
print(f"review length: median={int(np.median(lengths))}, max={max(lengths)}")

## Section 2 — Pad/truncate to a fixed length

Networks train on rectangular batches, but reviews have different lengths.

**TODO 1:** implement `pad_sequences(seqs, max_len)` returning an integer array of shape
`(len(seqs), max_len)` where each row is the sequence **truncated** to its last
`max_len` tokens (keep the END of the review) and **left-padded** with 0s if shorter.
Example with `max_len=4`: `[7, 8, 9]` → `[0, 7, 8, 9]`; `[3, 4, 5, 6, 7]` → `[4, 5, 6, 7]`.

In [ ]:
def pad_sequences(seqs, max_len=MAX_LEN):
    out = np.zeros((len(seqs), max_len), dtype=np.int64)
    # ===== TODO 1: fill `out` row by row (truncate to last max_len, left-pad with 0) =====
    raise NotImplementedError("TODO 1: implement pad_sequences, then delete this line.")

# Quick self-test (provided) — run it!
assert pad_sequences([[7, 8, 9]], 4).tolist() == [[0, 7, 8, 9]]
assert pad_sequences([[3, 4, 5, 6, 7]], 4).tolist() == [[4, 5, 6, 7]]
print("pad_sequences self-test passed ✔")

Xtr = pad_sequences(x_train); Xte = pad_sequences(x_test)
ytr = y_train.astype(np.float32); yte = y_test.astype(np.float32)
train_loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)),
                          batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(Xte), torch.tensor(yte)),
                         batch_size=256, shuffle=False)
print("Xtr:", Xtr.shape, " Xte:", Xte.shape)

## Section 3 — Two models: LSTM vs. bag-of-embeddings

**TODO 2 (LSTM classifier):**
* `__init__`: an `nn.Embedding(vocab_size, embed_dim, padding_idx=0)`, an
  `nn.LSTM(embed_dim, hidden_dim, batch_first=True)`, `nn.Dropout(0.3)`, and a final
  `nn.Linear(hidden_dim, 1)`.
* `forward(x)`: embed `x` → run the LSTM → take the **last hidden state** `h_n`
  (shape `(1, batch, hidden)` → squeeze to `(batch, hidden)`) → dropout → linear →
  return logits of shape `(batch,)` (use `.squeeze(-1)`).

**TODO 3 (baseline):** same embedding, but just **average** the embeddings over the
sequence dimension (`mean(dim=1)`), then dropout → linear. No recurrence at all.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        # ===== TODO 2a: embedding, lstm, dropout, fc =====
        pass  # replace with embedding, lstm, dropout, fc

    def forward(self, x):
        # ===== TODO 2b: embed -> lstm -> last hidden -> dropout -> fc -> squeeze =====
        raise NotImplementedError("TODO 2: implement the LSTM forward pass, then delete this line.")

class BagOfEmbeddings(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, embed_dim=EMBED_DIM):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # ===== TODO 3: embed -> mean over the sequence dimension -> dropout -> fc -> squeeze =====
        raise NotImplementedError("TODO 3: implement the baseline forward pass, then delete this line.")

# Shape check (provided).
with torch.no_grad():
    fake = torch.zeros(5, MAX_LEN, dtype=torch.long)
    assert LSTMClassifier()(fake).shape == (5,), "LSTM output should be (batch,)"
    assert BagOfEmbeddings()(fake).shape == (5,), "baseline output should be (batch,)"
print("shape checks passed ✔")

## Section 4 — Training step and evaluation

We use `nn.BCEWithLogitsLoss` (sigmoid + binary cross-entropy in one stable op).

**TODO 4:** one gradient step on a batch (move to device, zero grads, forward, loss,
backward, step, return `loss.item()`).

**TODO 5:** accuracy on a loader: a review is predicted positive when the **logit > 0**
(equivalently sigmoid > 0.5). Use `net.eval()` and `torch.no_grad()`.

In [ ]:
criterion = nn.BCEWithLogitsLoss()

def train_step(net, xb, yb, optimizer):
    net.train()
    # ===== TODO 4: implement the training step =====
    raise NotImplementedError("TODO 4: implement the training step, then delete this line.")

def evaluate(net, loader):
    net.eval()
    correct = total = 0
    # ===== TODO 5: count predictions where (logit > 0) matches the label =====
    raise NotImplementedError("TODO 5: implement evaluation, then delete this line.")

## Section 5 — Train and compare *(provided)*

In [ ]:
def fit(net, tag):
    net = net.to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    hist = []
    for epoch in range(1, EPOCHS + 1):
        losses = [train_step(net, xb, yb, opt) for xb, yb in train_loader]
        acc = evaluate(net, test_loader)
        hist.append((np.mean(losses), acc))
        print(f"[{tag}] epoch {epoch}/{EPOCHS}  loss={np.mean(losses):.4f}  test_acc={acc:.3f}")
    return net, hist

baseline, hist_base = fit(BagOfEmbeddings(), "baseline")
lstm, hist_lstm = fit(LSTMClassifier(), "LSTM")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot([h[0] for h in hist_base], label="baseline")
ax[0].plot([h[0] for h in hist_lstm], label="LSTM")
ax[0].set_title("training loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot([h[1] for h in hist_base], label="baseline")
ax[1].plot([h[1] for h in hist_lstm], label="LSTM")
ax[1].set_title("test accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## Section 6 — Where does recurrence struggle? *(provided)*

We split the test set into **short** and **long** reviews and score each group. Long
sequences are where vanilla recurrent models tend to lose information — the gradient
has to survive hundreds of steps. (Module 5 shows how attention removes this bottleneck.)

In [ ]:
true_lens = np.array([min(len(s), MAX_LEN) for s in x_test])
for name, net in [("baseline", baseline), ("LSTM", lstm)]:
    for tag, mask in [("short (<120 tokens)", true_lens < 120), ("long (>=120 tokens)", true_lens >= 120)]:
        ds = TensorDataset(torch.tensor(Xte[mask]), torch.tensor(yte[mask]))
        acc = evaluate(net, DataLoader(ds, batch_size=256))
        print(f"{name:9s} | {tag:20s} | n={mask.sum():5d} | acc={acc:.3f}")

In [ ]:
# Try the model on your own review (provided — edit the text!).
def predict_sentiment(text):
    ids = [1] + [word_index.get(w, -1) + 3 for w in text.lower().split()]  # unknown words -> id 2
    ids = [i if 0 <= i < VOCAB_SIZE else 2 for i in ids]
    x = torch.tensor(pad_sequences([ids])).to(device)
    lstm.eval()
    with torch.no_grad():
        p = torch.sigmoid(lstm(x)).item()
    return p

for review in ["this movie was an absolute joy to watch the acting was wonderful",
               "what a waste of two hours the plot made no sense at all"]:
    p = predict_sentiment(review)
    print(f"P(positive) = {p:.2f} | {review!r}")

## Checklist before you submit

* [ ] All cells run top-to-bottom without errors, outputs visible.
* [ ] The `pad_sequences` and shape self-tests pass.
* [ ] The LSTM reaches at least **~85%** test accuracy (baseline ~80%).
* [ ] The short-vs-long analysis and your own-review predictions are shown.
* [ ] In a final markdown cell (2-4 sentences): where did the LSTM beat the baseline,
      and what does the long-review result suggest about recurrent models?

**Submit your completed notebook (.ipynb) with all outputs visible.**